# Regime Filterの比較実験

**Historical archive / 過去の研究記録**

原本のコードを保持しています。独立実行や現在の検証基準への適合は保証しません。前のセルの変数に依存する箇所があります。実行入口は `../08_trade_quality.ipynb` を参照してください。

保存出力は `../../results/legacy/`、既知の問題は `../../docs/AUDIT.md` に整理しています。


## 元Notebookのセル 15

出典: `FX.ipynb`、0始まりのindex=14。コード内容は変更していません。

In [ ]:
# ============================================================
# 30分モデル + Regime Filter
#
# 目的：
#
# 1. 通常の二段階AI
#       vs
# 2. 得意な相場だけ取引する二段階AI
#
# を完全Walk-Forwardで比較する
#
# Regimeとして見るもの：
# ・ボラティリティ
# ・ADX
# ・MA50の傾き
#
# ============================================================


# ============================================================
# 1. ライブラリ
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 2. 基本設定
# ============================================================

HORIZON_BARS = 6

HORIZON_NAME = "30 MIN"

N_SPLITS_REGIME = 5

HALF_LIFE_DAYS_REGIME = 20


# 最低取引数
#
# Validationで数件だけ勝った設定を
# 「最適」と判断しないため
MIN_VALIDATION_TRADES = 20


# MOVE確率候補
MOVE_THRESHOLD_LIST = [
    0.55,
    0.60,
    0.65,
    0.70
]


# 方向確率候補
DIRECTION_THRESHOLD_LIST = [
    0.55,
    0.60,
    0.65,
    0.70
]


# TP候補
TP_LIST_REGIME = [
    0.0005,   # 0.05%
    0.0008,   # 0.08%
    0.0010    # 0.10%
]


# SL候補
SL_LIST_REGIME = [
    0.0005,
    0.0007,
    0.0010
]


# ------------------------------------------------------------
# Regime候補
# ------------------------------------------------------------

# ボラティリティの何％～何％だけ取引するか
VOL_RANGES = [

    (0.00, 1.00),   # 実質フィルタなし

    (0.10, 0.90),

    (0.20, 0.80),

    (0.30, 0.80)
]


# ADX最低値
ADX_MIN_LIST = [

    0,      # ADXフィルタなし

    20,

    25,

    30
]


# MA50 slope絶対値の
# 分位点以上を「トレンド」とする
SLOPE_QUANTILES = [

    0.00,   # slopeフィルタなし

    0.30,

    0.50
]


# ============================================================
# 3. ATRを計算
# ============================================================

previous_close = (
    df["Close"]
    .shift(1)
)


tr1 = (
    df["High"]
    - df["Low"]
)


tr2 = abs(
    df["High"]
    - previous_close
)


tr3 = abs(
    df["Low"]
    - previous_close
)


true_range = pd.concat(

    [
        tr1,
        tr2,
        tr3
    ],

    axis=1

).max(
    axis=1
)


df["ATR14"] = (

    true_range
    .rolling(14)
    .mean()
)


df["ATR14_pct"] = (

    df["ATR14"]
    / df["Close"]
)


# ============================================================
# 4. ADXを計算
# ============================================================

high_diff = (
    df["High"]
    .diff()
)


low_diff = (
    -df["Low"]
    .diff()
)


plus_dm = np.where(

    (
        high_diff > low_diff
    )

    &

    (
        high_diff > 0
    ),

    high_diff,

    0.0
)


minus_dm = np.where(

    (
        low_diff > high_diff
    )

    &

    (
        low_diff > 0
    ),

    low_diff,

    0.0
)


plus_dm = pd.Series(

    plus_dm,

    index=df.index
)


minus_dm = pd.Series(

    minus_dm,

    index=df.index
)


atr_for_adx = (

    true_range
    .rolling(14)
    .mean()
)


plus_di = (

    100

    * plus_dm
    .rolling(14)
    .mean()

    / atr_for_adx
)


minus_di = (

    100

    * minus_dm
    .rolling(14)
    .mean()

    / atr_for_adx
)


dx = (

    100

    * abs(
        plus_di
        - minus_di
    )

    /

    (
        plus_di
        + minus_di
    )
)


df["ADX14"] = (

    dx
    .rolling(14)
    .mean()
)


# ============================================================
# 5. 30分用データ作成
# ============================================================

data = (
    make_horizon_data(
        HORIZON_BARS
    )
)


# ATR / ADXを追加
data["ATR14_pct"] = (
    df.loc[
        data.index,
        "ATR14_pct"
    ]
)


data["ADX14"] = (
    df.loc[
        data.index,
        "ADX14"
    ]
)


data = (

    data
    .dropna()
    .copy()
)


print(
    "使用可能データ:",
    len(data)
)


# ============================================================
# 6. Profit Factor関数
# ============================================================

def profit_factor_function(
    returns
):

    returns = np.array(
        returns
    )


    positive = (

        returns[
            returns > 0
        ]
        .sum()
    )


    negative = abs(

        returns[
            returns < 0
        ]
        .sum()
    )


    if negative == 0:

        return np.nan


    return (
        positive
        / negative
    )


# ============================================================
# 7. シグナルをバックテストする関数
# ============================================================

def backtest_signals(

    frame,

    signals,

    horizon_bars,

    tp,

    sl

):


    returns = []

    directions = []

    times = []

    mfe_list = []

    mae_list = []


    i = 0


    while i < len(
        frame
    ):


        signal = (
            signals[i]
        )


        # 何もしない
        if signal == 0:

            i += 1

            continue


        time = (
            frame.index[i]
        )


        if signal == 1:

            direction = "BUY"

        else:

            direction = "SELL"


        r, mfe, mae = simulate_trade(

            signal_time=
                time,

            direction=
                direction,

            horizon_bars=
                horizon_bars,

            tp=
                tp,

            sl=
                sl
        )


        if not np.isnan(r):

            returns.append(
                r
            )

            directions.append(
                direction
            )

            times.append(
                time
            )

            mfe_list.append(
                mfe
            )

            mae_list.append(
                mae
            )


        # ポジション保有中は
        # 次のシグナルを無視
        i += horizon_bars


    return {

        "returns":
            np.array(
                returns
            ),

        "directions":
            directions,

        "times":
            times,

        "mfe":
            np.array(
                mfe_list
            ),

        "mae":
            np.array(
                mae_list
            )
    }


# ============================================================
# 8. 売買シグナル作成関数
# ============================================================

def make_signals(

    p_move,

    p_up,

    p_down,

    move_threshold,

    direction_threshold,

    regime_mask=None

):


    if regime_mask is None:

        regime_mask = np.ones(

            len(p_move),

            dtype=bool
        )


    buy = (

        regime_mask

        &

        (
            p_move
            >= move_threshold
        )

        &

        (
            p_up
            >= direction_threshold
        )

        &

        (
            p_up
            > p_down
        )
    )


    sell = (

        regime_mask

        &

        (
            p_move
            >= move_threshold
        )

        &

        (
            p_down
            >= direction_threshold
        )

        &

        (
            p_down
            > p_up
        )
    )


    signals = np.zeros(

        len(p_move)
    )


    signals[
        buy
    ] = 1


    signals[
        sell
    ] = -1


    return signals


# ============================================================
# 9. Regime Maskを作る関数
# ============================================================

def make_regime_mask(

    frame,

    vol_low_value,

    vol_high_value,

    adx_min,

    slope_min_value

):


    mask = (

        (
            frame[
                "volatility_1h"
            ].values
            >= vol_low_value
        )

        &

        (
            frame[
                "volatility_1h"
            ].values
            <= vol_high_value
        )

        &

        (
            frame[
                "ADX14"
            ].values
            >= adx_min
        )

        &

        (
            abs(
                frame[
                    "MA50_slope"
                ].values
            )
            >= slope_min_value
        )
    )


    return mask


# ============================================================
# 10. Walk-Forward開始
# ============================================================

block_size = (

    len(data)

    // (
        N_SPLITS_REGIME + 1
    )
)


fold_comparison = []

base_all_returns = []

regime_all_returns = []

base_all_times = []

regime_all_times = []


# ============================================================
# 11. Foldループ
# ============================================================

for fold in range(
    N_SPLITS_REGIME
):


    print(
        "\n\n================================"
    )

    print(
        "Fold",
        fold + 1
    )

    print(
        "================================"
    )


    # --------------------------------------------------------
    # Outer Train / Test
    # --------------------------------------------------------

    train_end = (

        block_size
        * (
            fold + 1
        )
    )


    test_start = (

        train_end
        + HORIZON_BARS
    )


    test_end = (

        test_start
        + block_size
    )


    if test_end > len(data):

        test_end = len(data)


    train_full = (

        data.iloc[
            :train_end
        ]
    )


    test = (

        data.iloc[
            test_start:test_end
        ]
    )


    if len(test) == 0:

        continue


    # --------------------------------------------------------
    # Inner Train / Validation
    # --------------------------------------------------------

    split_inner = int(

        len(train_full)
        * 0.80

    )


    train_core_end = (

        split_inner
        - HORIZON_BARS
    )


    train_core = (

        train_full.iloc[
            :train_core_end
        ]
    )


    validation = (

        train_full.iloc[
            split_inner:
        ]
    )


    if (

        len(train_core) < 300

        or

        len(validation) < 100

    ):

        continue


    # ========================================================
    # 12. MOVEモデル
    # ========================================================

    move_weights = (
        make_time_weights(

            train_core.index,

            HALF_LIFE_DAYS_REGIME
        )
    )


    move_model = (

        RandomForestClassifier(

            n_estimators=400,

            max_depth=8,

            min_samples_leaf=20,

            max_features="sqrt",

            class_weight="balanced",

            random_state=42,

            n_jobs=-1
        )
    )


    move_model.fit(

        train_core[
            move_features
        ],

        train_core[
            "move_target"
        ],

        sample_weight=
            move_weights
    )


    # ========================================================
    # 13. Directionモデル
    # ========================================================

    direction_train = (

        train_core[

            train_core[
                "move_target"
            ] == 1

        ]
    )


    if len(
        direction_train
    ) < 100:

        continue


    direction_weights = (

        make_time_weights(

            direction_train.index,

            HALF_LIFE_DAYS_REGIME
        )
    )


    direction_model = (

        RandomForestClassifier(

            n_estimators=400,

            max_depth=8,

            min_samples_leaf=15,

            max_features="sqrt",

            class_weight="balanced",

            random_state=42,

            n_jobs=-1
        )
    )


    direction_model.fit(

        direction_train[
            direction_features
        ],

        direction_train[
            "direction_target"
        ],

        sample_weight=
            direction_weights
    )


    # ========================================================
    # 14. Validation予測
    # ========================================================

    val_p_move = (

        move_model
        .predict_proba(

            validation[
                move_features
            ]

        )[:, 1]
    )


    val_dir_prob = (

        direction_model
        .predict_proba(

            validation[
                direction_features
            ]

        )
    )


    class_map = {

        c: i

        for i, c

        in enumerate(
            direction_model.classes_
        )
    }


    val_p_down = (

        val_dir_prob[
            :,
            class_map[0]
        ]
    )


    val_p_up = (

        val_dir_prob[
            :,
            class_map[1]
        ]
    )


    # ========================================================
    # 15. まずRegimeなし戦略を最適化
    # ========================================================

    best_base_score = -999

    best_base_settings = None


    for move_t in MOVE_THRESHOLD_LIST:

        for direction_t in DIRECTION_THRESHOLD_LIST:

            for tp in TP_LIST_REGIME:

                for sl in SL_LIST_REGIME:


                    signals = make_signals(

                        p_move=
                            val_p_move,

                        p_up=
                            val_p_up,

                        p_down=
                            val_p_down,

                        move_threshold=
                            move_t,

                        direction_threshold=
                            direction_t
                    )


                    bt = backtest_signals(

                        validation,

                        signals,

                        HORIZON_BARS,

                        tp,

                        sl
                    )


                    r = (
                        bt[
                            "returns"
                        ]
                    )


                    if len(
                        r
                    ) < MIN_VALIDATION_TRADES:

                        continue


                    # ----------------------------------------
                    # 平均期待値だけだと
                    # 数件の偶然を選びやすい
                    #
                    # そこで取引数も少し評価する
                    # ----------------------------------------

                    score = (

                        r.mean()

                        * np.sqrt(
                            len(r)
                        )
                    )


                    if score > best_base_score:

                        best_base_score = score


                        best_base_settings = {

                            "move_threshold":
                                move_t,

                            "direction_threshold":
                                direction_t,

                            "tp":
                                tp,

                            "sl":
                                sl
                        }


    if best_base_settings is None:

        print(
            "Base設定を選べません"
        )

        continue


    print(
        "Base設定:",
        best_base_settings
    )


    # ========================================================
    # 16. Base設定を固定してRegimeを探す
    # ========================================================

    best_regime_score = -999

    best_regime_settings = None


    # --------------------------------------------------------
    # Regime閾値はTrainだけから作る
    # --------------------------------------------------------

    for vol_q_low, vol_q_high in VOL_RANGES:


        vol_low_value = (

            train_core[
                "volatility_1h"
            ]
            .quantile(
                vol_q_low
            )
        )


        vol_high_value = (

            train_core[
                "volatility_1h"
            ]
            .quantile(
                vol_q_high
            )
        )


        for adx_min in ADX_MIN_LIST:


            for slope_q in SLOPE_QUANTILES:


                slope_min_value = (

                    abs(
                        train_core[
                            "MA50_slope"
                        ]
                    )

                    .quantile(
                        slope_q
                    )
                )


                regime_mask = make_regime_mask(

                    validation,

                    vol_low_value,

                    vol_high_value,

                    adx_min,

                    slope_min_value
                )


                signals = make_signals(

                    val_p_move,

                    val_p_up,

                    val_p_down,

                    best_base_settings[
                        "move_threshold"
                    ],

                    best_base_settings[
                        "direction_threshold"
                    ],

                    regime_mask=
                        regime_mask
                )


                bt = backtest_signals(

                    validation,

                    signals,

                    HORIZON_BARS,

                    best_base_settings[
                        "tp"
                    ],

                    best_base_settings[
                        "sl"
                    ]
                )


                r = (
                    bt[
                        "returns"
                    ]
                )


                if len(
                    r
                ) < MIN_VALIDATION_TRADES:

                    continue


                score = (

                    r.mean()

                    * np.sqrt(
                        len(r)
                    )
                )


                if score > best_regime_score:


                    best_regime_score = score


                    best_regime_settings = {

                        "vol_q_low":
                            vol_q_low,

                        "vol_q_high":
                            vol_q_high,

                        "vol_low":
                            vol_low_value,

                        "vol_high":
                            vol_high_value,

                        "adx_min":
                            adx_min,

                        "slope_q":
                            slope_q,

                        "slope_min":
                            slope_min_value
                    }


    # Regimeが選べない場合は
    # フィルタなし
    if best_regime_settings is None:


        best_regime_settings = {

            "vol_q_low":
                0,

            "vol_q_high":
                1,

            "vol_low":
                -np.inf,

            "vol_high":
                np.inf,

            "adx_min":
                0,

            "slope_q":
                0,

            "slope_min":
                0
        }


    print(
        "Regime設定:",
        best_regime_settings
    )


    # ========================================================
    # 17. 完全未知Test予測
    # ========================================================

    test_p_move = (

        move_model
        .predict_proba(

            test[
                move_features
            ]

        )[:, 1]
    )


    test_dir_prob = (

        direction_model
        .predict_proba(

            test[
                direction_features
            ]

        )
    )


    test_p_down = (

        test_dir_prob[
            :,
            class_map[0]
        ]
    )


    test_p_up = (

        test_dir_prob[
            :,
            class_map[1]
        ]
    )


    # ========================================================
    # 18. RegimeなしTest
    # ========================================================

    base_signals = make_signals(

        test_p_move,

        test_p_up,

        test_p_down,

        best_base_settings[
            "move_threshold"
        ],

        best_base_settings[
            "direction_threshold"
        ]
    )


    base_bt = backtest_signals(

        test,

        base_signals,

        HORIZON_BARS,

        best_base_settings[
            "tp"
        ],

        best_base_settings[
            "sl"
        ]
    )


    base_returns = (

        base_bt[
            "returns"
        ]
    )


    # ========================================================
    # 19. RegimeありTest
    # ========================================================

    regime_test_mask = make_regime_mask(

        test,

        best_regime_settings[
            "vol_low"
        ],

        best_regime_settings[
            "vol_high"
        ],

        best_regime_settings[
            "adx_min"
        ],

        best_regime_settings[
            "slope_min"
        ]
    )


    regime_signals = make_signals(

        test_p_move,

        test_p_up,

        test_p_down,

        best_base_settings[
            "move_threshold"
        ],

        best_base_settings[
            "direction_threshold"
        ],

        regime_mask=
            regime_test_mask
    )


    regime_bt = backtest_signals(

        test,

        regime_signals,

        HORIZON_BARS,

        best_base_settings[
            "tp"
        ],

        best_base_settings[
            "sl"
        ]
    )


    regime_returns = (

        regime_bt[
            "returns"
        ]
    )


    # ========================================================
    # 20. Fold統計
    # ========================================================

    def create_stats(
        returns
    ):


        if len(
            returns
        ) == 0:


            return {

                "trades":
                    0,

                "win_rate":
                    np.nan,

                "avg_return":
                    np.nan,

                "profit_factor":
                    np.nan
            }


        return {

            "trades":
                len(
                    returns
                ),

            "win_rate":
                (
                    returns > 0
                ).mean(),

            "avg_return":
                returns.mean(),

            "profit_factor":
                profit_factor_function(
                    returns
                )
        }


    base_stats = (
        create_stats(
            base_returns
        )
    )


    regime_stats = (
        create_stats(
            regime_returns
        )
    )


    print(
        "\nBASE:",
        base_stats
    )


    print(
        "REGIME:",
        regime_stats
    )


    # ========================================================
    # 21. 保存
    # ========================================================

    fold_comparison.append({

        "fold":
            fold + 1,

        "base_trades":
            base_stats[
                "trades"
            ],

        "base_win_rate":
            base_stats[
                "win_rate"
            ],

        "base_avg_return":
            base_stats[
                "avg_return"
            ],

        "base_pf":
            base_stats[
                "profit_factor"
            ],

        "regime_trades":
            regime_stats[
                "trades"
            ],

        "regime_win_rate":
            regime_stats[
                "win_rate"
            ],

        "regime_avg_return":
            regime_stats[
                "avg_return"
            ],

        "regime_pf":
            regime_stats[
                "profit_factor"
            ],

        "vol_low_quantile":
            best_regime_settings[
                "vol_q_low"
            ],

        "vol_high_quantile":
            best_regime_settings[
                "vol_q_high"
            ],

        "adx_min":
            best_regime_settings[
                "adx_min"
            ],

        "slope_quantile":
            best_regime_settings[
                "slope_q"
            ]
    })


    base_all_returns.extend(
        base_returns
    )


    regime_all_returns.extend(
        regime_returns
    )


    base_all_times.extend(
        base_bt[
            "times"
        ]
    )


    regime_all_times.extend(
        regime_bt[
            "times"
        ]
    )


# ============================================================
# 22. Fold比較
# ============================================================

comparison_df = pd.DataFrame(
    fold_comparison
)


percent_cols = [

    "base_win_rate",
    "base_avg_return",

    "regime_win_rate",
    "regime_avg_return"
]


for col in percent_cols:

    comparison_df[
        col
    ] *= 100


print(
    "\n\n===================================="
)

print(
    "Regimeなし vs Regimeあり Fold比較"
)

print(
    "===================================="
)


print(
    comparison_df
)


# ============================================================
# 23. 総合比較
# ============================================================

base_returns = np.array(
    base_all_returns
)


regime_returns = np.array(
    regime_all_returns
)


def overall_stats(
    returns
):


    if len(
        returns
    ) == 0:

        return {}


    equity = (

        1
        + pd.Series(
            returns
        )

    ).cumprod()


    running_max = (
        equity
        .cummax()
    )


    drawdown = (

        equity

        / running_max

        - 1
    )


    return {

        "trades":
            len(
                returns
            ),

        "win_rate":
            (
                returns > 0
            ).mean()
            * 100,

        "average_return":
            returns.mean()
            * 100,

        "profit_factor":
            profit_factor_function(
                returns
            ),

        "max_drawdown":
            drawdown.min()
            * 100,

        "total_growth":
            (
                equity.iloc[-1]
                - 1
            )
            * 100
    }


base_total = (
    overall_stats(
        base_returns
    )
)


regime_total = (
    overall_stats(
        regime_returns
    )
)


final_comparison = (
    pd.DataFrame(

        [

            {
                "strategy":
                    "BASE",

                **base_total
            },

            {
                "strategy":
                    "REGIME FILTER",

                **regime_total
            }

        ]

    )
)


print(
    "\n\n===================================="
)

print(
    "最終比較"
)

print(
    "===================================="
)


print(
    final_comparison
)


# ============================================================
# 24. Equity Curve比較
# ============================================================

if (
    len(base_returns) > 0
    and
    len(regime_returns) > 0
):


    base_equity = (

        1
        + pd.Series(
            base_returns
        )

    ).cumprod()


    regime_equity = (

        1
        + pd.Series(
            regime_returns
        )

    ).cumprod()


    plt.figure(
        figsize=(13, 6)
    )


    plt.plot(

        base_equity.values,

        label="BASE"
    )


    plt.plot(

        regime_equity.values,

        label="REGIME FILTER"
    )


    plt.xlabel(
        "Trade Number"
    )


    plt.ylabel(
        "Growth of 1"
    )


    plt.title(
        "Base vs Regime Filter"
    )


    plt.legend()

    plt.grid()

    plt.show()